# MyoInteract

Configure an interactive musculoskeletal task in a web UI, train a policy with mjlab (GPU), and preview it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoInteract/MyoInteract/blob/main/tutorials/myointeract/myoInteract.ipynb)

* **Colab:** use a GPU runtime (*Runtime → Change runtime type → T4 GPU*), then run the cells top to bottom. A short run takes under 20 minutes on a T4.
* **Local:** `pip install "myoInteract[all] @ git+https://github.com/MyoInteract/MyoInteract.git@v1.0.0"`, then open this notebook (or run `myo_play` for the UI alone).
* Logs are written locally and can be viewed with tensorboard. Weights & Biases is optional (set `LOGGER = "wandb"` below).


In [ ]:
# Colab: install MyoInteract (a few minutes). Locally this cell does nothing if it is already installed.
import importlib.util, sys

if importlib.util.find_spec("myoInteract") is None or (
    "google.colab" in sys.modules and importlib.util.find_spec("mjlab") is None
):
    %pip install -q "myoInteract[colab] @ git+https://github.com/MyoInteract/MyoInteract.git@v1.0.0"
    if "google.colab" in sys.modules:
        print("Installed. If the next cell fails to import, restart the runtime (Runtime -> Restart session) and run again from here.")


In [ ]:
## Name of this experiment, and where training is logged: "tensorboard" (local, default) or "wandb".
PROJECT_NAME = "myoInteract-test"
LOGGER = "tensorboard"
WANDB_ENTITY = None  # only used with LOGGER = "wandb" (None = your default entity)

import os

os.environ["MYOINTERACT_LOGGER"] = LOGGER
if LOGGER == "wandb":
    import wandb


In [ ]:
from myoInteract.scripts.ui import get_ui, RunState

state = RunState()
demo = get_ui(PROJECT_NAME, state)
demo.launch(share=True, height=800)


### Optional: render a policy loaded via the UI

After using **Load / Resume from Checkpoint** in the UI above (and optionally loading the training config), click **Update Preview** in the UI so the loaded config is reflected, then run the cell below to render the loaded policy in the viser viewer — without launching a new training run.

In [ ]:
from pathlib import Path
from IPython.display import display, IFrame

if not state.loaded_checkpoint_path:
    print("No checkpoint loaded yet. Use 'Load / Resume from Checkpoint' in the UI above.")
else:
    # state.cfg_overrides is authoritative once rebuilt from the UI (e.g. via
    # "Update Preview" or "Run Training" above) -- see the note in
    # ui.py's render_training_result() for why a separately-cached snapshot
    # of the loaded policy's overrides is not used here.
    url = state.render_training_result(
        log_dir=str(Path(state.loaded_checkpoint_path).parent)
    )
    print(f"Policy: {Path(state.loaded_checkpoint_path).name}")
    display(IFrame(url, width="100%", height=600))

In [ ]:
from myoInteract.myointeract_hydra_cli import load_config_interactive

config = load_config_interactive(state.cfg_overrides)
state.cfg_overrides

### Training

Execute the following cell to run the training:

In [ ]:
from myoInteract.scripts.ui import launch_training

if LOGGER == "wandb":
    wandb.init(project=PROJECT_NAME, name=state.wandb_run_name, entity=WANDB_ENTITY)
launch_training(state)


In [ ]:
# Training curves (tensorboard) or the wandb run page.
from IPython.display import display, IFrame

if LOGGER == "wandb":
    if wandb.run is not None:
        print(f"WandB run: {wandb.run.url}")
        display(IFrame(src=wandb.run.url, width="100%", height="800"))
else:
    %load_ext tensorboard
    %tensorboard --logdir {state.log_dir}


In [ ]:
# Keep the result: zip of the run folder (config + checkpoints). On Colab this starts a download.
import shutil

archive = shutil.make_archive(f"{PROJECT_NAME}_run", "zip", state.log_dir)
print("Saved", archive)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    pass
